In [0]:

import uuid
from datetime import datetime, timezone

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType,
    TimestampType, LongType
)

# Audit configuration
PIPELINE_NAME = "retail_data_pipeline"
TASK_NAME = "03_silver_transformation"
AUDIT_TABLE = "workspace.retail_training.pipeline_audit_log"

RUN_ID = str(uuid.uuid4())
START_TIME = datetime.now(timezone.utc)

ROWS_PROCESSED = 0

print(f"Task audit ID: {RUN_ID}")
print("Silver transformation configuration loaded.")


Task audit ID: 89a4aab9-494e-426a-9c7b-f52ed4714bd6
Silver transformation configuration loaded.


In [0]:

def write_audit_record(
    run_status,
    start_time,
    end_time,
    rows_processed=None,
    error_message=None
):
    audit_schema = StructType([
        StructField("pipeline_name", StringType(), False),
        StructField("task_name", StringType(), False),
        StructField("run_id", StringType(), True),
        StructField("run_status", StringType(), False),
        StructField("start_time", TimestampType(), True),
        StructField("end_time", TimestampType(), True),
        StructField("rows_processed", LongType(), True),
        StructField("error_message", StringType(), True),
        StructField("logged_at", TimestampType(), True)
    ])

    audit_record = [(
        PIPELINE_NAME,
        TASK_NAME,
        RUN_ID,
        run_status,
        start_time,
        end_time,
        rows_processed,
        error_message[:4000] if error_message else None,
        end_time
    )]

    audit_df = spark.createDataFrame(
        audit_record,
        schema=audit_schema
    )

    audit_df.write.format("delta").mode("append").saveAsTable(
        AUDIT_TABLE
    )


In [0]:

try:
    # 1. Read Bronze sales and dimension tables
    bronze_sales = spark.table(
        "workspace.retail_training.bronze_sales"
    )

    customers = spark.table(
        "workspace.retail_training.bronze_customers"
    )

    products = spark.table(
        "workspace.retail_training.bronze_products"
    )

    stores = spark.table(
        "workspace.retail_training.bronze_stores"
    )

    # 2. Remove rows with missing sale IDs
    sales_with_ids = bronze_sales.filter(
        F.col("sale_id").isNotNull()
    )

    # 3. Keep only valid customer, product, and store references
    valid_sales = (
        sales_with_ids
        .join(
            customers.select("customer_id").distinct(),
            on="customer_id",
            how="left_semi"
        )
        .join(
            products.select("product_id").distinct(),
            on="product_id",
            how="left_semi"
        )
        .join(
            stores.select("store_id").distinct(),
            on="store_id",
            how="left_semi"
        )
    )

    # 4. Standardize data types and calculate revenue
    silver_sales = (
        valid_sales
        .withColumn("sale_id", F.col("sale_id").cast("long"))
        .withColumn("customer_id", F.col("customer_id").cast("long"))
        .withColumn("product_id", F.col("product_id").cast("long"))
        .withColumn("store_id", F.col("store_id").cast("long"))
        .withColumn("quantity", F.col("quantity").cast("int"))
        .withColumn(
            "unit_price",
            F.col("unit_price").cast("decimal(12,2)")
        )
        .withColumn("sale_date", F.to_date("sale_date"))
        .withColumn(
            "revenue",
            (F.col("quantity") * F.col("unit_price"))
            .cast("decimal(14,2)")
        )
        .withColumn("_silver_processed_at", F.current_timestamp())
    )

    # 5. Remove duplicate sale IDs
    silver_sales = silver_sales.dropDuplicates(["sale_id"])

    # 6. Validate critical fields before writing
    invalid_rows = silver_sales.filter(
        F.col("sale_id").isNull()
        | F.col("customer_id").isNull()
        | F.col("product_id").isNull()
        | F.col("store_id").isNull()
        | F.col("quantity").isNull()
        | (F.col("quantity") <= 0)
        | F.col("unit_price").isNull()
        | (F.col("unit_price") < 0)
        | F.col("sale_date").isNull()
        | F.col("revenue").isNull()
    ).count()

    if invalid_rows > 0:
        raise RuntimeError(
            f"Silver validation failed: {invalid_rows} invalid rows found."
        )

    # 7. Count records before writing
    silver_count = silver_sales.count()
    ROWS_PROCESSED = silver_count

    # 8. Save Silver sales using the existing full-refresh approach
    (
        silver_sales.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable("workspace.retail_training.silver_sales")
    )

    print("Silver transformation completed.")
    print(f"Silver rows: {silver_count}")
    print("SUCCESS: Silver data passed the critical-field checks.")

except Exception as e:
    end_time = datetime.now(timezone.utc)

    try:
        write_audit_record(
            run_status="FAILED",
            start_time=START_TIME,
            end_time=end_time,
            rows_processed=ROWS_PROCESSED,
            error_message=str(e)
        )
        print("FAILED audit record written.")
    except Exception as audit_error:
        print(f"WARNING: Could not write failure audit record: {audit_error}")

    raise

else:
    end_time = datetime.now(timezone.utc)

    write_audit_record(
        run_status="SUCCESS",
        start_time=START_TIME,
        end_time=end_time,
        rows_processed=ROWS_PROCESSED,
        error_message=None
    )

    print(f"SUCCESS audit record written: {ROWS_PROCESSED} Silver rows.")


Silver transformation completed.
Silver rows: 110
SUCCESS: Silver data passed the critical-field checks.
SUCCESS audit record written: 110 Silver rows.
